In [ ]:
# Import Libraries
import math
import pandas as pd
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import Model
from tensorflow.keras import Sequential
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from tensorflow.keras.losses import MeanSquaredLogarithmicError
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Load Data
data = pd.read_csv('fulldata.csv', index_col = 'Date')
data.shape

In [ ]:
#Drop Sentiment Data
data = data.drop('vader_polarity', axis=1)

In [ ]:
# Read over ARIMA predictions
%store -r full_arima_predictions

# Increment Indices
arima_predictions_list = full_arima_predictions.tolist()
arima_predictions_list.append(arima_predictions_list.pop(0))

# Append ARIMA predictions to dataframe
data['arima'] = arima_predictions_list

# Append close to dataframe
data['y'] = data['close_aapl']

In [ ]:
#Data preview
data

In [ ]:
data.shape

In [ ]:
# Convert data to numpy array
dataset = data.values

# Get training set size 80:20 split
training_data_len = math.ceil(len(dataset) * 0.8)
training_data_len

In [ ]:
# Scaler for data
scaler = MinMaxScaler(feature_range=(0,1))

In [ ]:
# Create training dataset
train_data = dataset[0:training_data_len,:]

# Split to x_train and y_train
x_train = train_data[:,:50]
y_train = train_data[:,50]

# Create test dataset
test_data = dataset[training_data_len:,:]

# Split to x_test and y_test
x_test = test_data[:,:50]
y_test = test_data[:,50]

# Show shapes of sets
print('x_train shape: ' + str(x_train.shape))
print('y_train shape: ' + str(y_train.shape))
print()
print('x_test shape: ' + str(x_test.shape))
print('y_test shape: ' + str(y_test.shape))

In [ ]:
# Convert training and testing sets to numpy arrays
x_train = np.array(x_train)
y_train = np.array(y_train)

x_test = np.array(x_test)
y_test = np.array(y_test)

# Scale data
scaled_data = scaler.fit_transform(dataset)

x_train = scaler.fit_transform(x_train)
y_train = scaler.fit_transform(y_train.reshape(-1,1))
x_test = scaler.fit_transform(x_test)
y_test = scaler.fit_transform(y_test.reshape(-1,1))

x_train

In [ ]:
# Train Split Preview
print(x_train)
print(x_test)

# Test Split Preview
print(y_train)
print(y_test)

In [ ]:
# Build autoencoder
# Code modified from: https://www.analyticsvidhya.com/blog/2021/06/dimensionality-reduction-using-autoencoders-in-python/
class AutoEncoders(Model):

    # Initializer
    def __init__(self, output_units):

        super().__init__()

        # Build Encoder down to 10 features
        self.encoder = Sequential(
            [
                Dense(50, activation="relu"),
                Dense(37, activation="relu"),
                Dense(25, activation="relu")
            ]
        )

        # Build Decoder
        self.decoder = Sequential(
            [
                Dense(37, activation="relu"),
                Dense(51, activation="relu"),
                Dense(output_units, activation="sigmoid")
            ]
        )

    # Call Overwrite
    def call(self, inputs):
        # Encode
        encoded = self.encoder(inputs)
        
        # Decode
        decoded = self.decoder(encoded)
        
        # Return
        return decoded
    
# Autoencoder Usage
    # Instantiate Autoencoder
auto_encoder = AutoEncoders(50)
    
    # Prep for training
auto_encoder.compile( loss='mae', metrics=['mae'], optimizer='adam')

    # Train
history = auto_encoder.fit(
    x_train, 
    x_train, 
    epochs=250, 
    batch_size=16, 
    validation_data=(x_test, x_test)
)


In [ ]:
print(history.history.keys())

In [ ]:
# Visualize Loss over Epochs
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
# Use encoder to compress feature set
encoder_layer = auto_encoder.get_layer('sequential_30')
compressed_x_train = encoder_layer.predict(x_train)
compressed_x_test = encoder_layer.predict(x_test)

In [ ]:
reduced_x_train = pd.DataFrame(compressed_x_train)
reduced_x_train = reduced_x_train.add_prefix('feature_')
reduced_x_train

In [ ]:
reduced_x_test = pd.DataFrame(compressed_x_test)
reduced_x_test = reduced_x_test.add_prefix('feature_')
reduced_x_test

In [ ]:
# Store compressed feature set for use by wGAN-GP
%store compressed_x_train
%store compressed_x_test